# Práctica 5: Fine-tuning de un Detector de Spam con DistilBERT

En este cuaderno implementamos el ajuste fino (*fine-tuning*) del modelo `distilbert-base-uncased`
para la tarea de **clasificación de texto** (detección de spam en mensajes SMS).

Utilizamos el dataset [SMS Spam Collection](https://huggingface.co/datasets/sms_spam) (~5.5k mensajes)
y seguimos la metodología del curso de Hugging Face y la clase de Lingüística Computacional.

Al final, el modelo entrenado se sube al Hub de Hugging Face y se despliega con una interfaz Gradio.

## 1. Instalación de dependencias

Ejecuta esta celda si estás en Google Colab o Kaggle.

In [1]:
!pip install -q datasets transformers evaluate accelerate huggingface_hub gradio scikit-learn codecarbon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.5/380.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 123.9 MB/s eta 0:00:00


## 2. Carga del Dataset SMS Spam Collection

El dataset contiene ~5,574 mensajes SMS etiquetados como `ham` (legítimo) o `spam`.

In [2]:
from datasets import load_dataset

# Using the standard dataset identifier
raw_dataset = load_dataset("ucirvine/sms_spam")
print(raw_dataset)

README.md:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})


In [3]:
# Explorar un ejemplo
print(raw_dataset["train"][0])
print(f"\nCaracterísticas: {raw_dataset['train'].features}")

{'sms': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n', 'label': 0}

Características: {'sms': Value('string'), 'label': ClassLabel(names=['ham', 'spam'])}


### 2.1 División del dataset

El dataset solo tiene una partición `train`. Lo dividimos en train/validation/test (80/10/10).

In [4]:
# Primera división: 80% train, 20% temp
train_temp = raw_dataset["train"].train_test_split(test_size=0.2, seed=42, stratify_by_column="label")

# Segunda división: dividir temp en 50/50 para val y test (10% cada uno del total)
val_test = train_temp["test"].train_test_split(test_size=0.5, seed=42, stratify_by_column="label")

from datasets import DatasetDict

split_dataset = DatasetDict({
    "train": train_temp["train"],
    "validation": val_test["train"],
    "test": val_test["test"],
})

print(split_dataset)

# Distribución de clases
for split_name in split_dataset:
    labels = split_dataset[split_name]["label"]
    n_spam = sum(labels)
    n_ham = len(labels) - n_spam
    print(f"  {split_name}: {len(labels)} total | ham={n_ham} | spam={n_spam}")

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 4459
    })
    validation: Dataset({
        features: ['sms', 'label'],
        num_rows: 557
    })
    test: Dataset({
        features: ['sms', 'label'],
        num_rows: 558
    })
})
  train: 4459 total | ham=3861 | spam=598
  validation: 557 total | ham=483 | spam=74
  test: 558 total | ham=483 | spam=75


## 3. Tokenización

Usamos el tokenizador de `distilbert-base-uncased` con truncamiento a 512 tokens.

In [5]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Ejemplo de tokenización
example = split_dataset["train"][0]
tokens = tokenizer(example["sms"], truncation=True, max_length=512)
print(f"Texto: {example['sms'][:80]}...")
print(f"Tokens (primeros 10): {tokens['input_ids'][:10]}")
print(f"Total tokens: {len(tokens['input_ids'])}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Texto: Dont you have message offer
...
Tokens (primeros 10): [101, 2123, 2102, 2017, 2031, 4471, 3749, 102]
Total tokens: 8


In [6]:
def tokenize_function(examples):
    return tokenizer(examples["sms"], truncation=True, max_length=512)


tokenized_dataset = split_dataset.map(tokenize_function, batched=True)
print(tokenized_dataset)

Map:   0%|          | 0/4459 [00:00<?, ? examples/s]

Map:   0%|          | 0/557 [00:00<?, ? examples/s]

Map:   0%|          | 0/558 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sms', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4459
    })
    validation: Dataset({
        features: ['sms', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 557
    })
    test: Dataset({
        features: ['sms', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 558
    })
})


### 3.1 Data Collator con Padding Dinámico

Usamos `DataCollatorWithPadding` para aplicar padding dinámico por batch,
lo cual es más eficiente que padding global.

In [7]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 4. Métricas de Evaluación

Definimos una función que calcula **accuracy**, **F1**, **precision** y **recall**.

In [8]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    prec = precision_metric.compute(predictions=predictions, references=labels)
    rec = recall_metric.compute(predictions=predictions, references=labels)

    return {
        "accuracy": acc["accuracy"],
        "f1": f1["f1"],
        "precision": prec["precision"],
        "recall": rec["recall"],
    }

## 5. Autenticación en Hugging Face

Ingresa tu token de escritura (*Write* token) para poder subir el modelo al Hub.

In [9]:
from huggingface_hub import notebook_login

notebook_login()

## 6. Carga del Modelo y Configuración del Entrenamiento

Cargamos `distilbert-base-uncased` con 2 etiquetas (ham/spam).

In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2,
    id2label={0: "ham", 1: "spam"},
    label2id={"ham": 0, "spam": 1},
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="distilbert-spam-detector",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    push_to_hub=True,
    report_to="none",
)

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 7. Entrenamiento

In [13]:
from codecarbon import EmissionsTracker

tracker = EmissionsTracker()
tracker.start()
try:
    trainer.train()
finally:
    tracker.stop()

[codecarbon WARNING @ 00:04:36] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 00:04:37] [setup] RAM Tracking...
[codecarbon INFO @ 00:04:37] [setup] CPU Tracking...
[codecarbon WARNING @ 00:04:38] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 00:04:38] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 00:04:38] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon WARNING @ 00:04:38] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 00:04:38] [setup] GPU Tracking...
[codecarbon INFO @ 00:04:38] Tracking Nvidia GPUs via PyNVML
[codecarbon INFO @ 00:04:38] The below tracking methods have been set up:
                RAM Tracking Method: RAM p

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.019730,0.996409,0.986301,1.000000,0.972973
2,No log,0.016516,0.996409,0.986301,1.000000,0.972973
3,No log,0.017024,0.996409,0.986301,1.000000,0.972973


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[codecarbon INFO @ 00:04:53] Energy consumed for RAM : 0.000083 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 00:04:53] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 00:04:53] Energy consumed for All CPU : 0.000177 kWh
[codecarbon INFO @ 00:04:53] Monitoring GPUs with indices: [0] out of 1 total GPUs
[codecarbon INFO @ 00:04:53] Energy consumed for all GPUs : 0.000261 kWh. Total GPU Power : 62.55111964380074 W
[codecarbon INFO @ 00:04:53] 0.000521 kWh of electricity and 0.000000 L of water were used since the beginning.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
[codecarbon INFO @ 00:05:08] Energy consumed for RAM : 0.000167 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 00:05:08] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 00:05:08] Energy consumed for All CPU : 0.000354 kWh
[codecarbon INFO @ 00:05:08] Energy consumed for all GPUs : 0.000463 kWh. Total GPU Power : 48.67203030548803 W
[codecarbon INFO @ 00:05:08] 0.000984 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 00:05:10] Energy consumed for RAM : 0.000181 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 00:05:10] Delta energy consumed for CPU with constant : 0.000030 kWh, power : 42.5 W
[codecarbon INFO @ 00:05:10] Energy co

## 8. Evaluación en el Conjunto de Test

Evaluamos el modelo final en los datos de test (no vistos durante entrenamiento ni validación).

In [14]:
test_results = trainer.evaluate(tokenized_dataset["test"])
print("Resultados en Test:")
for k, v in test_results.items():
    if k.startswith("eval_"):
        print(f"  {k.replace('eval_', '')}: {v:.4f}" if isinstance(v, float) else f"  {k.replace('eval_', '')}: {v}")

Resultados en Test:
  loss: 0.0556
  accuracy: 0.9857
  f1: 0.9459
  precision: 0.9589
  recall: 0.9333
  runtime: 0.2409
  samples_per_second: 2316.7560
  steps_per_second: 37.3670


## 9. Prueba de Inferencia Local

Probamos el modelo con algunos mensajes de ejemplo.

In [18]:
from transformers import pipeline

classifier = pipeline("text-classification", model="distilbert-spam-detector", tokenizer=tokenizer)

test_messages = [
    "Congratulations! You've won a free iPhone. Click here to claim now!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account has been compromised. Call this number immediately.",
    "Can you pick up some milk on your way home?",
    "FREE entry to WIN £1000 cash! Text WIN to 12345 now!",
    "I'll be there in 10 minutes, just parking the car.",
]

print("--- Pruebas de Detección de Spam ---")
for msg in test_messages:
    result = classifier(msg)[0]
    emoji = "🚫" if result["label"] == "spam" else "✅"
    print(f"{emoji} [{result['label'].upper():>4}] ({result['score']:.3f}) {msg[:70]}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

--- Pruebas de Detección de Spam ---
🚫 [SPAM] (0.911) Congratulations! You've won a free iPhone. Click here to claim now!
✅ [ HAM] (0.995) Hey, are we still meeting for lunch tomorrow?
✅ [ HAM] (0.965) URGENT: Your account has been compromised. Call this number immediatel
✅ [ HAM] (0.994) Can you pick up some milk on your way home?
🚫 [SPAM] (0.986) FREE entry to WIN £1000 cash! Text WIN to 12345 now!
✅ [ HAM] (0.996) I'll be there in 10 minutes, just parking the car.


## 10. Subir Modelo al Hub

In [16]:
trainer.push_to_hub(
    tags=["text-classification", "spam-detection"],
    commit_message="Fine-tuned DistilBERT spam detector",
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...etector/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...etector/model.safetensors:  45%|####4     |  120MB /  268MB            

CommitInfo(commit_url='https://huggingface.co/toporaku/distilbert-spam-detector/commit/1531ba349b47dc121260d2d109d8c8162c9117fc', commit_message='Fine-tuned DistilBERT spam detector', commit_description='', oid='1531ba349b47dc121260d2d109d8c8162c9117fc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/toporaku/distilbert-spam-detector', endpoint='https://huggingface.co', repo_type='model', repo_id='toporaku/distilbert-spam-detector'), pr_revision=None, pr_num=None)

## 11. Demo con Gradio

Lanzamos una interfaz interactiva para probar el detector de spam.
Con `share=True` se genera una URL pública temporal.

In [17]:
import gradio as gr

# Recargar pipeline desde el directorio local
spam_classifier = pipeline("text-classification", model="distilbert-spam-detector", tokenizer=tokenizer)


def detect_spam(text):
    if not text or not text.strip():
        return {}
    result = spam_classifier(text)[0]
    label = result["label"]
    score = result["score"]
    # Devolver ambas probabilidades
    if label == "spam":
        return {"spam 🚫": score, "ham ✅": 1 - score}
    else:
        return {"ham ✅": score, "spam 🚫": 1 - score}


demo = gr.Interface(
    fn=detect_spam,
    inputs=gr.Textbox(
        label="Mensaje SMS",
        placeholder="Escribe o pega un mensaje para analizar...",
        lines=3,
    ),
    outputs=gr.Label(label="Clasificación", num_top_classes=2),
    title="🔍 Detector de Spam en SMS",
    description="Modelo DistilBERT ajustado con el corpus SMS Spam Collection. Escribe un mensaje y descubre si es spam o no.",
    examples=[
        ["Congratulations! You've won a free trip. Reply YES to claim!"],
        ["Hey, can you send me the notes from today's class?"],
        ["WINNER!! You have been selected for a cash prize of £5000!"],
        ["I'll pick you up at 7, see you soon."],
    ],
    theme=gr.themes.Soft(),
)

demo.launch(share=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b91bc800b17cf96d07.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
